# UniCompose3D 1.15.1 · Kaggle Studio

Reconstruct objects with SAM and Hunyuan3D, then compose them in an interactive 3D editor using GPT-6 Luna.

Attach **UniCompose3D.zip** as Input, enable **T4 GPU + Internet**, and add the **OPENAI_API_KEY** Secret. Run **1 → 2 → 2B → 3 → 4**. No HF token is required for the standard setup.

Weights need approximately 11.2 GiB. Start with at least 14 GiB free disk and no competing GPU task. Setup and startup do not call paid APIs. Keep the complete session link private.

Single-image reconstruction infers hidden geometry. Automatic composition supports reusable spatial relations but cannot guarantee success for every object pair. Save projects before replacing source or restarting.

## 1. Prepare source

Attach the release ZIP as Input. Restart Session before loading an updated source version.

In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, shutil, sys, zipfile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
PROJECT = WORK / "UniCompose3D"
EXPECTED_VERSION = "1.15.1"
if "unicompose3d" in sys.modules or "launch" in sys.modules:
    raise RuntimeError("Source modules are already imported. Restart Session after updating, then run from section 1.")
ARCHIVE_NAME = "UniCompose3D.zip"
archives = sorted(INPUT.rglob(ARCHIVE_NAME))
extracted = sorted(p.parent for p in INPUT.rglob("launch.py") if (p.parent / "unicompose3d/api.py").is_file())
if len(archives) > 1 or (not archives and len(extracted) > 1):
    raise RuntimeError("Multiple source datasets found. Keep only the Input containing version 1.15.1.")
if archives:
    source_zip = archives[0]
    with zipfile.ZipFile(source_zip) as z:
        if sum(i.file_size for i in z.infolist()) > 100 * 1024 * 1024:
            raise RuntimeError("Source ZIP exceeds the size limit. Check the uploaded file.")
        for item in z.infolist():
            relative = PurePosixPath(item.filename)
            if relative.is_absolute() or ".." in relative.parts or not relative.parts or relative.parts[0] != "UniCompose3D" or "\\" in item.filename:
                raise RuntimeError("ZIP contains an invalid path.")
            target = WORK.joinpath(*relative.parts).resolve()
            if not target.is_relative_to(PROJECT.resolve()):
                raise RuntimeError("ZIP path escapes the project directory.")
        z.extractall(WORK)
    print("Extracted:", source_zip.name)
elif extracted:
    shutil.copytree(extracted[0], PROJECT, dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__", "data", ".venv", "test-results"))
    print("Copied the source extracted by Kaggle.")
else:
    raise FileNotFoundError("UniCompose3D.zip or extracted source was not found. Attach the source dataset and rerun.")
version_source = (PROJECT / "unicompose3d/__init__.py").read_text(encoding="utf-8")
if ('__version__ = "' + EXPECTED_VERSION + '"') not in version_source:
    raise RuntimeError("Source version does not match " + EXPECTED_VERSION + ". Update the source dataset and restart the session.")
manifest_path = PROJECT / "unicompose3d/static/vendor/manifest.json"
if not manifest_path.exists():
    raise RuntimeError("Three.js assets are missing. Use a complete release bundle.")
manifest = json.loads(manifest_path.read_text())
for name, expected in manifest["sha256"].items():
    vendor_file = manifest_path.parent / name
    if not vendor_file.exists() or hashlib.sha256(vendor_file.read_bytes()).hexdigest() != expected:
        raise RuntimeError("Missing or corrupt browser asset: " + name)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print("Source ready at:", PROJECT)
print("Version:", EXPECTED_VERSION)
print("Python:", sys.version.split()[0], "No API calls made.")

## 2. Install Studio dependencies

Wait for completion. If Kaggle requests a restart, restart and rerun from section 1.

In [ ]:
import importlib, subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "-r", str(PROJECT / "requirements.txt")])
importlib.invalidate_caches()
print("Studio installed. Run section 2B to install SAM and Hunyuan Turbo.")

## 2B. Install Hunyuan3D and SAM

The first setup downloads weights and compiles extensions. Subsequent calls reuse the environment.

In [ ]:
import subprocess, sys, os
from pathlib import Path
BACKEND = "hunyuan"
os.environ["TRYON_DEFAULT_PROVIDER"] = "hunyuan"
subprocess.check_call([sys.executable, "-u", str(PROJECT / "tools/setup_quality.py")])
os.environ["TRYON_HUNYUAN_PYTHON"] = "/kaggle/working/tryon-quality-env/bin/python"
os.environ["TRYON_HUNYUAN_DIR"] = "/kaggle/working/Hunyuan3D-2"
os.environ["TRYON_QUALITY_WEIGHTS"] = "/kaggle/working/tryon-quality-weights"
os.environ["TRYON_SEGMENT_PYTHON"] = os.environ["TRYON_HUNYUAN_PYTHON"]
os.environ["TRYON_SEGMENT_WEIGHTS"] = os.environ["TRYON_QUALITY_WEIGHTS"]
print("Hunyuan3D and SAM are ready. Continue with sections 3 and 4.")

## 3. Read Secrets and configure

Enable notebook access to OPENAI_API_KEY. Keys stay in the Python environment and are never printed. API model access and quota depend on your account.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets_client = UserSecretsClient()
for key in ("OPENAI_API_KEY",):
    try:
        value = secrets_client.get_secret(key).strip()
        if value:
            os.environ[key] = value
    except Exception:
        pass
    print(key + ": " + ("available" if os.environ.get(key) else "missing - add the Secret to enable the corresponding feature"))
os.environ["TRYON_LLM_PROVIDER"] = "openai"  # Optional alternative: gemini
os.environ["OPENAI_MODEL"] = "gpt-6-luna"
os.environ["OPENAI_COMPOSITION_REASONING_EFFORT"] = "medium"
os.environ["TRYON_SEARCH_MAX_STARTS"] = "16"
os.environ["TRYON_SEARCH_MAX_PROGRAMS"] = "4"
os.environ["TRYON_SEARCH_MAX_SECONDS"] = "60"
os.environ["TRYON_DATA_DIR"] = "/kaggle/working/tryon-data"
print("LLM:", os.environ["TRYON_LLM_PROVIDER"], "· OpenAI:", os.environ["OPENAI_MODEL"])
print("Keys are never printed. Hunyuan3D uses the Kaggle GPU; OpenAI handles localization and prompt-based composition.")

## 4. Launch Studio

Open the complete link, including #token=..., in another tab. Keep this notebook running.

In [ ]:
from launch import start, studio_share_again
studio = start(share=True, port=7860, data_dir=os.environ["TRYON_DATA_DIR"])
from IPython.display import HTML, display
import html
display(HTML('<a target="_blank" rel="noopener noreferrer" style="font-size:18px" href="' + html.escape(studio.url, quote=True) + '">Open UniCompose3D Studio ↗</a>'))

## Use the Studio

1. Add A/B using images, video frames, or existing models.
2. Select the intended object. Draw a box for fast SAM segmentation or describe it for optional AI localization. Refine masks and confirm every selected view.
3. Reconstruct with Hunyuan3D. Multiview input requires up to four explicitly assigned camera views. Review the shape before choosing Use as A/B.
4. Enter a short request such as "Place the vase on the table." Run Compose with agent and answer any clarification.
5. Review proposals, adjust transforms if needed, then download GLB or project ZIP.

Composition reuses the meshes and may call the LLM multiple times within its round limit. It does not reconstruct the objects again. Results serve visual exploration, not physical sizing or guaranteed fit.

## Optional: reconnect the public link

Only use when the server is running but its tunnel failed. No reconstruction or LLM calls.

In [ ]:
# Uncomment to retry the tunnel.
# studio_share_again()

## Optional: back up all session data

Wait for active jobs to finish first. Download the resulting archive before ending the session.

In [ ]:
# Uncomment both lines to back up your data.
# backup_path = shutil.make_archive("/kaggle/working/UniCompose3D_backup", "zip", root_dir=os.environ["TRYON_DATA_DIR"])
# print("Created:", backup_path, "- download this file from the Kaggle Output/Files panel.")

## Optional: stop Studio

Export projects or back up the data directory before ending Kaggle.

In [ ]:
# Uncomment to stop the server. Rerun Secrets and section 4 to restart.
# studio.stop()